# Cheatsheet: Multiple Linear Regression for Economic / Econometric Analysis
### Companion reference for the "Population + Income → Profit" lab (and any similar project)

This notebook is a **reference you run alongside your own project**, not a graded exercise.
Every formula below has a matching, runnable code cell using a tiny toy dataset, so you can see
inputs/outputs immediately rather than just reading math.

# Contents
- [ 1 - Notation cheat sheet ](#1)
- [ 2 - Core formulas (model, cost, gradient) ](#2)
- [ 3 - Vectorization patterns (loop → NumPy) ](#3)
- [ 4 - Feature scaling reference ](#4)
- [ 5 - Gradient descent hyperparameter troubleshooting ](#5)
- [ 6 - Closed-form OLS vs. gradient descent ](#6)
- [ 7 - Economic interpretation reference ](#7)
- [ 8 - Model diagnostics reference ](#8)
- [ 9 - Common bugs and how to spot them ](#9)
- [ 10 - Quick-reference function library ](#10)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

# Tiny toy dataset used for every worked example below (3 examples, 2 features)
X_toy = np.array([
    [10.0, 40.0],
    [15.0, 60.0],
    [20.0, 50.0],
])
y_toy = np.array([8.0, 15.0, 14.0])
print("X_toy:\n", X_toy)
print("y_toy:", y_toy)


<a name="1"></a>
## 1 - Notation Cheat Sheet

| Symbol | Meaning | Shape | Economic reading |
|---|---|---|---|
| $m$ | number of training examples | scalar | number of cities/observations |
| $n$ | number of features | scalar | number of predictors (e.g., 2: population, income) |
| $X$ | design matrix | `(m, n)` | dataset table, rows=observations, cols=variables |
| $\mathbf{x}^{(i)}$ | feature vector for example $i$ | `(n,)` | one city's [population, income] |
| $x_j^{(i)}$ | value of feature $j$ for example $i$ | scalar | e.g., city 3's income |
| $y^{(i)}$ | target/label for example $i$ | scalar | that city's observed profit |
| $\mathbf{w}$ | weight/coefficient vector | `(n,)` | marginal effects (per feature) |
| $w_j$ | coefficient for feature $j$ | scalar | "dependent variable" change per unit of $x_j$ |
| $b$ | bias / intercept | scalar | fitted value when all features = 0 |
| $f_{w,b}(x)$ | model prediction | scalar (per example) | predicted profit |
| $J(w,b)$ | cost function (mean squared error / 2) | scalar | how wrong the model is, overall |
| $\alpha$ | learning rate | scalar | step size in gradient descent |
| $\mu_j, \sigma_j$ | mean / std of feature $j$ | scalar each | used for z-score normalization |


<a name="2"></a>
## 2 - Core Formulas

**Model (prediction):**
$$f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w}\cdot\mathbf{x}^{(i)} + b = \sum_{j=1}^n w_j x_j^{(i)} + b$$

**Cost (mean squared error, halved):**
$$J(\mathbf{w},b) = \frac{1}{2m}\sum_{i=0}^{m-1}\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}\right)^2$$

**Gradients:**
$$\frac{\partial J}{\partial w_j} = \frac{1}{m}\sum_{i=0}^{m-1}\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)})-y^{(i)}\right)x_j^{(i)}
\qquad
\frac{\partial J}{\partial b} = \frac{1}{m}\sum_{i=0}^{m-1}\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)})-y^{(i)}\right)$$

**Parameter update (one gradient descent step):**
$$w_j \leftarrow w_j - \alpha\frac{\partial J}{\partial w_j} \qquad b \leftarrow b - \alpha\frac{\partial J}{\partial b}$$

**Why the 1/2?** It exists purely so the derivative's factor of 2 (from the power rule on the
square) cancels cleanly — it does not change *which* $(w,b)$ minimizes $J$, only the value of
$J$ itself.


In [ ]:
def f_predict(X, w, b):
    return X @ w + b

def compute_cost(X, y, w, b):
    m = X.shape[0]
    errors = f_predict(X, w, b) - y
    return (1/(2*m)) * np.sum(errors**2)

def compute_gradient(X, y, w, b):
    m = X.shape[0]
    errors = f_predict(X, w, b) - y
    dj_dw = (1/m) * (X.T @ errors)
    dj_db = (1/m) * np.sum(errors)
    return dj_dw, dj_db

w_demo, b_demo = np.array([0.5, 0.1]), 1.0
print("prediction:", f_predict(X_toy, w_demo, b_demo))
print("cost:", compute_cost(X_toy, y_toy, w_demo, b_demo))
dj_dw, dj_db = compute_gradient(X_toy, y_toy, w_demo, b_demo)
print("dj_dw:", dj_dw, " dj_db:", dj_db)


<a name="3"></a>
## 3 - Vectorization Patterns: Loop → NumPy

The single most common source of bugs when moving from 1 feature to $n$ features is writing
loops that should be matrix operations. Use this table to translate directly.

| Written out (don't do this for n>1) | Vectorized (do this) | Notes |
|---|---|---|
| `for i in range(m): pred[i] = sum(w[j]*X[i,j] for j in range(n)) + b` | `X @ w + b` | One matmul replaces the double loop |
| `for i in range(m): err[i] = pred[i]-y[i]` | `pred - y` | Elementwise subtraction, no loop |
| `total = sum(err[i]**2 for i in range(m)); cost = total/(2*m)` | `np.sum(err**2) / (2*m)` | — |
| `for j in range(n): dj_dw[j] = sum(err[i]*X[i,j] for i in range(m))/m` | `(X.T @ err) / m` | `X.T` is `(n,m)`, so this is `(n,m)@(m,) -> (n,)` |
| `dj_db = sum(err[i] for i in range(m))/m` | `np.sum(err) / m` | — |

**Shape-checking habit:** before trusting any vectorized line, mentally (or actually) print
`.shape` for every array involved. `X @ w` requires `X.shape = (m, n)` and `w.shape = (n,)` to
produce `(m,)` — a `(n,1)` shaped `w` will silently broadcast wrong and give a `(m,n)` result
instead of `(m,)`, which is one of the most common multi-feature bugs.


In [ ]:
# Demonstrating the shape trap
w_col = np.array([[0.5],[0.1]])   # WRONG shape: (2,1) instead of (2,)
w_flat = np.array([0.5, 0.1])     # RIGHT shape: (2,)

print("X_toy @ w_flat shape:", (X_toy @ w_flat).shape, " <- correct, this is what you want: (m,)")
print("X_toy @ w_col  shape:", (X_toy @ w_col).shape,  " <- WRONG, silently becomes (m,1), breaks (m,)-y subtraction")


<a name="4"></a>
## 4 - Feature Scaling Reference

| Method | Formula | When to use |
|---|---|---|
| Z-score normalization | $x_{norm} = \dfrac{x-\mu}{\sigma}$ | Default choice; works well with gradient descent |
| Min-max scaling | $x_{norm} = \dfrac{x-x_{min}}{x_{max}-x_{min}}$ | When you want bounded `[0,1]` features |
| Mean normalization | $x_{norm} = \dfrac{x-\mu}{x_{max}-x_{min}}$ | Less common; centers at 0 but bounded range |
| Log transform | $x_{norm} = \log(x)$ | Skewed economic variables (income, population, GDP) — also gives elasticity-style coefficients directly if $y$ is also logged |

**Rules that prevent data leakage:**
1. Compute $\mu,\sigma$ **only** on the training set.
2. Apply the *same* $\mu,\sigma$ to validation/test/new data — never recompute on them.
3. To report coefficients in original units, **un-scale after training**:
   $$w_{j,\text{raw}} = \frac{w_{j,\text{norm}}}{\sigma_j} \qquad
     b_{\text{raw}} = b_{\text{norm}} - \sum_j \frac{w_{j,\text{norm}}\mu_j}{\sigma_j}$$


In [ ]:
def zscore_normalize_features(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    return (X - mu) / sigma, mu, sigma

def unscale_coefficients(w_norm, b_norm, mu, sigma):
    w_raw = w_norm / sigma
    b_raw = b_norm - np.sum((w_norm * mu) / sigma)
    return w_raw, b_raw

X_norm, mu, sigma = zscore_normalize_features(X_toy)
print("X_norm:\n", X_norm)
w_raw, b_raw = unscale_coefficients(np.array([0.3, -0.2]), 12.0, mu, sigma)
print("un-scaled example: w_raw =", w_raw, " b_raw =", b_raw)


<a name="5"></a>
## 5 - Gradient Descent Hyperparameter Troubleshooting

| Symptom (watching `J_history`) | Likely cause | Fix |
|---|---|---|
| Cost decreases then increases, or oscillates wildly | `alpha` too large | Reduce `alpha` by 3-10x |
| Cost decreases extremely slowly, still falling a lot at last iteration | `alpha` too small, or not enough iterations | Increase `alpha` and/or `num_iters` |
| Cost is `nan` or `inf` after a few iterations | `alpha` way too large (divergence), or unscaled features with huge range | Scale features (Section 4) and/or shrink `alpha` drastically |
| Cost plateaus quickly at a high value | Model underfits — may need more/better features, or a nonlinear term | Add features (e.g., interaction/polynomial terms) |
| Different features have very different gradient magnitudes at iteration 0 | Features aren't scaled | Apply z-score normalization first |

**Practical recipe:** try `alpha` on a log scale: `0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0` —
plot `J_history` for each and pick the largest `alpha` that still converges smoothly.


In [ ]:
def gradient_descent(X, y, w_in, b_in, alpha, num_iters):
    import copy
    w, b = copy.deepcopy(w_in), b_in
    J_history = []
    for i in range(num_iters):
        dj_dw, dj_db = compute_gradient(X, y, w, b)
        w = w - alpha*dj_dw
        b = b - alpha*dj_db
        J_history.append(compute_cost(X, y, w, b))
    return w, b, J_history

fig, ax = plt.subplots(figsize=(6,4))
for alpha in [0.01, 0.1, 0.5]:
    _, _, Jh = gradient_descent(X_norm, y_toy, np.zeros(2), 0., alpha, 50)
    ax.plot(Jh, label=f"alpha={alpha}")
ax.set_xlabel("iteration"); ax.set_ylabel("cost J"); ax.legend(); ax.set_title("Learning-rate sweep (toy data)")
plt.show()


<a name="6"></a>
## 6 - Closed-Form OLS vs. Gradient Descent

For **plain linear regression only**, the exact minimizer of $J$ has a closed form:

$$ \hat{\boldsymbol{\theta}} = (X_b^T X_b)^{-1} X_b^T \mathbf{y} $$

where $X_b$ is $X$ with a column of 1's prepended (for the intercept) and
$\hat{\boldsymbol{\theta}} = [b, w_1, \dots, w_n]$.

| | Closed-form (normal equation) | Gradient descent |
|---|---|---|
| Exact or iterative? | Exact, one shot | Iterative, approaches optimum |
| Needs `alpha`/iterations? | No | Yes — must tune |
| Needs feature scaling? | No (numerically nicer with it, but not required for correctness) | Strongly recommended |
| Cost for large $n$ (many features) | Slow: inverting an $(n{+}1)\times(n{+}1)$ matrix is $O(n^3)$ | Fine: each step is $O(mn)$ |
| Generalizes beyond linear regression? | No | Yes — the same idea trains logistic regression, neural networks, etc. |

**Use the closed form as a sanity check** on small/medium problems, and gradient descent as the
technique that will actually scale and generalize to the more complex models you'll meet later.


In [ ]:
def ols_closed_form(X, y):
    m = X.shape[0]
    X_design = np.column_stack([np.ones(m), X])
    theta = np.linalg.lstsq(X_design, y, rcond=None)[0]
    return theta[0], theta[1:]   # b, w

b_ols, w_ols = ols_closed_form(X_toy, y_toy)
print("Closed-form: b =", b_ols, " w =", w_ols)

w_gd, b_gd, _ = gradient_descent(X_norm, y_toy, np.zeros(2), 0., alpha=0.3, num_iters=2000)
w_gd_raw, b_gd_raw = unscale_coefficients(w_gd, b_gd, mu, sigma)
print("Gradient descent (un-scaled): b =", b_gd_raw, " w =", w_gd_raw)
print("(On this 3-point toy dataset they won't match perfectly to 4 decimals with only",
      "2000 iterations and no assertion — that's expected on tiny/noisy toy data;",
      "on the full 100-city lab dataset the two methods agree closely.)")


<a name="7"></a>
## 7 - Economic Interpretation Reference

| Quantity | Formula | Interpretation |
|---|---|---|
| Raw coefficient $w_{j,\text{raw}}$ | (see Section 4 un-scaling) | Change in $y$ per 1-unit change in $x_j$, holding other features fixed ("marginal effect", ceteris paribus) |
| Standardized coefficient $w_{j,\text{norm}}$ | trained directly on z-scored $X$ | Change in $y$ per 1-**standard-deviation** change in $x_j$ — comparable across features regardless of original units |
| Elasticity | $w_{j,\text{raw}}\cdot \bar{x}_j/\bar{y}$ | % change in $y$ per 1% change in $x_j$, evaluated at the means — unit-free, classic economics reporting |
| Log-log elasticity (alternative) | fit $\ln y = w_1\ln x_1 + \dots + b$ | Then $w_j$ **is** the elasticity directly, constant across the whole range (not just at the mean) |
| $R^2$ | see Section 8 | Fraction of variance in $y$ explained by the model |

**Common pitfalls when writing up results:**
- Don't compare *raw* coefficients across features measured in different units — compare
  standardized coefficients or elasticities instead.
- A coefficient's sign and rough magnitude can be trusted; don't over-interpret the intercept
  $b$ if $x=0$ is outside the realistic data range (e.g., a city with 0 population).
- Correlation between features (multicollinearity) can make individual coefficients unstable
  even when the overall model fits well — always check a correlation matrix or scatter of the
  features against each other before trusting individual $w_j$ values.


In [ ]:
x_bar = np.mean(X_toy, axis=0)
y_bar = np.mean(y_toy)
elasticities = w_ols * (x_bar / y_bar)
print("Elasticities at the mean:", elasticities)
print("Feature correlation matrix:\n", np.corrcoef(X_toy.T))


<a name="8"></a>
## 8 - Model Diagnostics Reference

| Diagnostic | Formula / method | What to look for |
|---|---|---|
| $R^2$ | $1 - SS_{res}/SS_{tot}$ | Closer to 1 = more variance explained; compare across models cautiously (adding *any* feature can't decrease it) |
| Adjusted $R^2$ | $1-(1-R^2)\frac{m-1}{m-n-1}$ | Penalizes adding useless features — better for comparing models with different $n$ |
| Residual plot | scatter of residuals vs. fitted $\hat y$ | Should look like random noise around 0 — a funnel shape suggests heteroskedasticity; a curve suggests a missing nonlinear term |
| Actual vs. predicted plot | scatter of $y$ vs. $\hat y$ | Points should hug the 45° line |
| Correlation of features | `np.corrcoef` | High correlation between two features (e.g., >0.8) is a multicollinearity warning |


In [ ]:
def r_squared(y, y_pred):
    ss_res = np.sum((y - y_pred)**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    return 1 - ss_res/ss_tot

def adjusted_r_squared(y, y_pred, n_features):
    m = len(y)
    r2 = r_squared(y, y_pred)
    return 1 - (1 - r2) * (m - 1) / (m - n_features - 1)

y_pred_toy = f_predict(X_toy, w_ols, b_ols)
print("R^2:", r_squared(y_toy, y_pred_toy))
# NOTE: adjusted R^2 needs m > n+1; this 3-point/2-feature toy example is too small to
# demonstrate it meaningfully (division by ~0) -- use it on your real, larger dataset instead.


<a name="9"></a>
## 9 - Common Bugs and How to Spot Them

| Bug | Symptom | How to catch it |
|---|---|---|
| `w` has shape `(n,1)` instead of `(n,)` | Cost/gradient functions return arrays instead of scalars, or shapes silently broadcast to `(m,n)` | Print `.shape` after every major operation; use `np.zeros(n)` not `np.zeros((n,1))` |
| Forgot to divide by `m` in cost or gradient | Cost/gradients are `m` times too large; gradient descent diverges even at small `alpha` | Compare cost at `w=0,b=0` to a hand-calculated or known expected value |
| Applied normalization stats from test set instead of train set | Model looks great on paper but predictions on truly new data are inconsistent/wrong | Always store `mu`, `sigma` from training and reuse them everywhere downstream |
| Mixed normalized and raw features between training and prediction | Predictions are wildly off scale | Keep one clear boundary: train and predict *both* in normalized space, then only un-scale coefficients for reporting |
| Used `X * w` instead of `X @ w` | Wrong shape entirely (elementwise instead of matrix product), errors like "operands could not be broadcast" | Remember `@` for matrix/vector products; `*` is elementwise |
| Learning rate too high with unscaled features | Cost becomes `nan`/`inf` after a few iterations | Scale features first (Section 4), then tune `alpha` (Section 5) |


<a name="10"></a>
## 10 - Quick-Reference Function Library

Copy-paste-ready versions of everything above, already correct, for dropping into a new
project (or see the companion **reusable template notebook** which packages these into a
config-driven pipeline).

In [ ]:
import numpy as np

def compute_cost(X, y, w, b):
    m = X.shape[0]
    errors = X @ w + b - y
    return (1/(2*m)) * np.sum(errors**2)

def compute_gradient(X, y, w, b):
    m = X.shape[0]
    errors = X @ w + b - y
    dj_dw = (1/m) * (X.T @ errors)
    dj_db = (1/m) * np.sum(errors)
    return dj_dw, dj_db

def zscore_normalize_features(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    return (X - mu) / sigma, mu, sigma

def unscale_coefficients(w_norm, b_norm, mu, sigma):
    w_raw = w_norm / sigma
    b_raw = b_norm - np.sum((w_norm * mu) / sigma)
    return w_raw, b_raw

def gradient_descent(X, y, w_in, b_in, alpha, num_iters):
    import copy
    w, b = copy.deepcopy(w_in), b_in
    J_history = []
    for i in range(num_iters):
        dj_dw, dj_db = compute_gradient(X, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        J_history.append(compute_cost(X, y, w, b))
    return w, b, J_history

def r_squared(y, y_pred):
    ss_res = np.sum((y - y_pred)**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    return 1 - ss_res/ss_tot

def ols_closed_form(X, y):
    m = X.shape[0]
    X_design = np.column_stack([np.ones(m), X])
    theta = np.linalg.lstsq(X_design, y, rcond=None)[0]
    return theta[0], theta[1:]

print("Function library loaded — ready to copy into a new notebook or .py module.")
